# 1D MALA Drift Report

This notebook is a sequential report of one-dimensional MALA drift experiments. It uses the implementation in `src/mala_1d.py` and focuses, for each experiment, on:

- total drift
- regional drift contributions
- proposal probabilities
- accepted probabilities
- the diagnostic quantity `R(x)`

In [ ]:
import importlib.util
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

candidate_roots = [Path.cwd(), Path.cwd().parent]
repo_root = next(root for root in candidate_roots if (root / "src" / "mala_1d.py").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

module_path = repo_root / "src" / "mala_1d.py"
spec = importlib.util.spec_from_file_location("mala_1d", module_path)
mala_1d = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mala_1d)

## Shared Setup

All experiments are run over a fixed grid of `x` values rather than along a Markov chain trajectory. This keeps the diagnostics local in `x` and makes the drift decomposition easier to interpret.

In [ ]:
x_grid = np.linspace(0.5, 6.0, 30)
n_samples = 20_000
seed = 12345
eps = 0.1


def run_and_plot_experiment(p, h, lyapunov_type, **kwargs):
    lyapunov = mala_1d.build_lyapunov(
        lyapunov_type=lyapunov_type,
        p=p,
        **kwargs,
    )

    result = mala_1d.run_experiment(
        x_grid=x_grid,
        h=h,
        p=p,
        n_samples=n_samples,
        seed=seed,
        V_func=lyapunov["V_func"],
        log_V_func=lyapunov["log_V_func"],
        eps=eps,
    )

    mala_1d.plot_experiment(
        result=result,
        p=p,
        h=h,
        lyapunov_label=lyapunov["label"],
        eps=eps,
        output_path=None,
    )

    return result, lyapunov

## Experiment 1: `p=2`, `h=0.05`, `V(x)=1+|x|`

This is the baseline polynomial Lyapunov choice. It is useful for checking whether the negative contribution from the inward region `A1` is already visible before the proposal begins to overshoot substantially.

In [ ]:
result_1, lyapunov_1 = run_and_plot_experiment(
    p=2,
    h=0.05,
    lyapunov_type="poly",
    m=1.0,
)

For this baseline case, the total drift should become negative as `x` grows. The regional panel is the key one: we expect `A1` to provide the dominant negative contribution, `A2` to stay closer to zero, and `A3` to remain small. The probability panels should show that outward proposals are already relatively rare, while `R(x)` stays modest on this grid.

## Experiment 2: `p=2`, `h=0.05`, `V(x)=1+|x|^2`

This experiment increases the polynomial growth of the Lyapunov function. The goal is to see whether stronger growth amplifies the visible drift signal while still keeping the decomposition numerically stable.

In [ ]:
result_2, lyapunov_2 = run_and_plot_experiment(
    p=2,
    h=0.05,
    lyapunov_type="poly",
    m=2.0,
)

Compared with `V(x)=1+|x|`, the total drift curve should generally be more pronounced because large moves are weighted more heavily. The regional split still matters most: if `A1` remains the main negative term and `A3` stays negligible, then the mechanism is not changing, only its magnitude under the stronger Lyapunov weight.

## Experiment 3: `p=2`, `h=0.05`, `V(x)=\exp(0.02|x|)`

This is the first exponential Lyapunov experiment. It is mild enough to probe faster growth than the polynomial case without making the weighting too aggressive.

In [ ]:
result_3, lyapunov_3 = run_and_plot_experiment(
    p=2,
    h=0.05,
    lyapunov_type="exp",
    s=0.02,
    beta=1.0,
)

The exponential Lyapunov function places more emphasis on large-`|y|` proposals than the polynomial one. If the total drift remains negative mainly because of `A1`, while `A3` is still tiny in both proposal and accepted probability, then the qualitative drift mechanism is robust to this change of Lyapunov scale.

## Experiment 4: `p=2`, `h=0.05`, `V(x)=\exp(0.05|x|)`

This increases the exponential growth rate and tests whether the drift decomposition remains interpretable when outward moves are penalized more strongly through the Lyapunov function.

In [ ]:
result_4, lyapunov_4 = run_and_plot_experiment(
    p=2,
    h=0.05,
    lyapunov_type="exp",
    s=0.05,
    beta=1.0,
)

Here the total drift may react more sharply to rare large excursions, but the proposal and accepted probability panels help separate geometry from weighting. If `A3` continues to be negligible and the accepted outward mass collapses quickly, then the interesting story is not frequent outward escape, but rather the increasing mismatch between proposal scale and local acceptance.

## Experiment 5: `p=3`, `h=0.01`, `V(x)=1+|x|`

Now the target tail parameter is increased to `p=3`. The smaller step size is chosen to partially compensate, but the diagnostic `R(x)` becomes especially important because stronger gradients can produce overshoot much earlier.

In [ ]:
result_5, lyapunov_5 = run_and_plot_experiment(
    p=3,
    h=0.01,
    lyapunov_type="poly",
    m=1.0,
)

With larger `p`, the inward drift from the deterministic proposal term becomes stronger, but so does the risk of proposal overshoot. The most informative panels here are the accepted probabilities and `R(x)`: once accepted mass starts collapsing while `R(x)` rises, the obstruction is no longer just drift size but the inability of the proposal to remain in an acceptably local regime.

## Experiment 6: `p=3`, `h=0.01`, `V(x)=\exp(0.02|x|)`

This final experiment combines the steeper target with a mild exponential Lyapunov function. It is designed to make the rejection-collapse mechanism easier to see when outward moves become both costly and unlikely to be accepted.

In [ ]:
result_6, lyapunov_6 = run_and_plot_experiment(
    p=3,
    h=0.01,
    lyapunov_type="exp",
    s=0.02,
    beta=1.0,
)

This case is useful for checking that the same qualitative picture persists under a different Lyapunov scale. If the total drift is still largely explained by `A1`, while `A3` remains negligible and the accepted outward mass drops rapidly, then the main limitation is proposal overshoot followed by rejection collapse, not a failure of the Lyapunov weighting itself.

## Summary

Across these experiments, the most consistent picture is the following:

- `A1` dominates the negative drift.
- `A3` is negligible both at the proposal level and, even more strongly, after acceptance.
- Increasing `p` causes rejection collapse earlier, even when `h` is reduced.
- The main obstruction appears to be proposal overshoot followed by rejection collapse.

The diagnostic `R(x)` helps organize this interpretation. When `R(x)` grows, the proposal drift is becoming too large relative to the local scale set by `x`, and the accepted-probability panels then show how this turns into rejection collapse. In that regime, the limiting factor is not the existence of a negative inward contribution from `A1`, but whether the proposal remains local enough for acceptance to preserve a useful drift estimate.